In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("bundesligascsv/bundesligaseason-2425.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'bundesligascsv/bundesligaseason-2425.csv'

In [ ]:
df.dtypes

In [ ]:
df.head

In [ ]:
df.shape

In [ ]:
df["HomeTeam"].value_counts()

In [ ]:
home_teams = df['HomeTeam'].unique()
away_teams = df['AwayTeam'].unique()

In [ ]:
home_teams

In [ ]:
away_teams

In [ ]:
teams = pd.unique(pd.concat([pd.Series(home_teams), pd.Series(away_teams)]))

In [ ]:
import pandas as pd
import numpy as np

FILE_MAP = {
    "bundesligascsv/bundesligaseason-2425.csv": 2025,
    "bundesligascsv/bundesligaseason-2324.csv": 2024,
    "bundesligascsv/bundesligaseason-2223.csv": 2023,
    "bundesligascsv/bundesligaseason-2122.csv": 2022,
    "bundesligascsv/bundesligaseason-2021.csv": 2021
}

FINAL_COLS = [
    'date', 'day', 'comp', 'venue', 'result', 'gf', 'ga', 'opponent',
    'referee', 'sh', 'sot', 'foul_comm', 'foul_rec', 'yc_rec', 'rc_rec',
    'yc_opp', 'rc_opp', 'season', 'team', 'day_code'
]

def get_result(ftr, venue):
    if venue == 'Home':
        if ftr == 'H': return 'W'
        elif ftr == 'A': return 'L'
        else: return 'D'
    elif venue == 'Away':
        if ftr == 'A': return 'W'
        elif ftr == 'H': return 'L'
        else: return 'D'
    return None

def process_season_data(file_name, season_year):
    print(f"Processing {file_name}...")
    try:
        season_df = pd.read_csv(file_name)

        season_df['date'] = pd.to_datetime(season_df['Date'], format='%d/%m/%y').dt.strftime('%Y-%m-%d')
        season_df['season'] = season_year

        home_df = pd.DataFrame({
            'date': season_df['date'],
            'team': season_df['HomeTeam'],
            'opponent': season_df['AwayTeam'],
            'venue': 'Home',
            'gf': season_df['FTHG'],
            'ga': season_df['FTAG'],
            'result': season_df.apply(lambda row: get_result(row['FTR'], 'Home'), axis=1),
            'sh': season_df['HS'],
            'sot': season_df['HST'],
            'referee': season_df['Referee'],
            'season': season_df['season'],
            'foul_comm': season_df['HF'],
            'foul_rec': season_df['AF'],
            'yc_rec': season_df['HY'],
            'rc_rec': season_df['HR'],
            'yc_opp': season_df['AY'],
            'rc_opp': season_df['AR']
        })

        away_df = pd.DataFrame({
            'date': season_df['date'],
            'team': season_df['AwayTeam'],
            'opponent': season_df['HomeTeam'],
            'venue': 'Away',
            'gf': season_df['FTAG'],
            'ga': season_df['FTHG'],
            'result': season_df.apply(lambda row: get_result(row['FTR'], 'Away'), axis=1),
            'sh': season_df['AS'],
            'sot': season_df['AST'],
            'referee': season_df['Referee'],
            'season': season_df['season'],
            'foul_comm': season_df['AF'],
            'foul_rec': season_df['HF'],
            'yc_rec': season_df['AY'],
            'rc_rec': season_df['AR'],
            'yc_opp': season_df['HY'],
            'rc_opp': season_df['HR']
        })

        combined_df = pd.concat([home_df, away_df], ignore_index=True)

        combined_df['comp'] = 'Bundesliga'
        combined_df['date_dt'] = pd.to_datetime(combined_df['date'])
        combined_df['day'] = combined_df['date_dt'].dt.day_name().str[:3]
        combined_df['day_code'] = combined_df['date_dt'].dt.dayofweek
        combined_df = combined_df.drop(columns=['date_dt'])

        float_cols = [
            'gf', 'ga', 'sh', 'sot', 'foul_comm', 'foul_rec', 'yc_rec', 'rc_rec', 'yc_opp', 'rc_opp'
        ]
        for col in float_cols:
            combined_df[col] = combined_df[col].astype(float)

        final_df = combined_df[FINAL_COLS].copy()

        return final_df
    except Exception as e:
        print(f"Error processing file {file_name}: {e}")
        return pd.DataFrame(columns=FINAL_COLS)

all_seasons_data = []

for file_name, season_year in FILE_MAP.items():
    processed_df = process_season_data(file_name, season_year)
    all_seasons_data.append(processed_df)

combined_data = pd.concat(all_seasons_data, ignore_index=True)

combined_data.sort_values(by=['date', 'team'], inplace=True, ignore_index=True)

combined_file_name = 'bundesligascsv/bundesligaall_seasons_combined_sorted.csv'
combined_data.to_csv(combined_file_name, index=False)

print(f"\nSuccessfully combined all data into: {combined_file_name}")
print(f"Total rows in combined data: {len(combined_data)}")

In [ ]:
combined_data

In [ ]:
combined_data["venue_code"] = combined_data["venue"].astype("category").cat.codes

In [ ]:
combined_data["opp_code"] = combined_data["opponent"].astype("category").cat.codes

In [ ]:
combined_data["day_code"] = pd.to_datetime(combined_data["date"]).dt.dayofweek

In [ ]:
combined_data["target"] = (combined_data["result"] == "W").astype("int")

In [ ]:
combined_data

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators = 40, min_samples_split = 300, random_state=1)

In [ ]:
train = combined_data[combined_data["date"] < '2023-01-01']

In [ ]:
test = combined_data[combined_data["date"] > '2023-01-01']

In [ ]:
predictors = ["venue_code", "opp_code", "day_code"]

In [8]:
rf.fit(train[predictors], train["target"])

NameError: name 'rf' is not defined

In [9]:
preds = rf.predict(test[predictors])

NameError: name 'rf' is not defined

In [10]:
from sklearn.metrics import accuracy_score

In [11]:
acc = accuracy_score(test["target"], preds)

NameError: name 'test' is not defined

In [27]:
acc

0.6289910600255428

In [28]:
combined = pd.DataFrame(dict(actual=test["target"], prediction=preds))

In [29]:
pd.crosstab(index=combined["actual"], columns=combined["prediction"])

prediction,0,1
actual,,
0,975,10
1,571,10


In [30]:
from sklearn.metrics import precision_score

In [31]:
precision_score(test["target"], preds)

0.5

In [35]:
grouped_matches = combined_data.groupby("team")

In [36]:
group = grouped_matches.get_group("Bayern Munich")

In [37]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [38]:
cols = ["gf","ga","sh","sot"]
new_cols = [f"{c}_rolling" for c in cols]

In [39]:
new_cols

['gf_rolling', 'ga_rolling', 'sh_rolling', 'sot_rolling']

In [40]:
combined_data.dtypes

date           object
day            object
comp           object
venue          object
result         object
gf            float64
ga            float64
opponent       object
referee       float64
sh            float64
sot           float64
foul_comm     float64
foul_rec      float64
yc_rec        float64
rc_rec        float64
yc_opp        float64
rc_opp        float64
season          int64
team           object
day_code        int32
venue_code       int8
opp_code         int8
target          int64
dtype: object

In [41]:
rolling_averages(group,cols,new_cols)

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
55,2020-10-17,Sat,Bundesliga,Away,W,4.0,1.0,Bielefeld,NaN,16.0,...,2021,Bayern Munich,5,0,2,1,4.333333,2.333333,19.333333,7.666667
74,2020-10-24,Sat,Bundesliga,Home,W,5.0,0.0,Ein Frankfurt,NaN,20.0,...,2021,Bayern Munich,5,1,6,1,3.000000,2.666667,17.333333,7.000000
93,2020-10-31,Sat,Bundesliga,Away,W,2.0,1.0,FC Koln,NaN,7.0,...,2021,Bayern Munich,5,0,7,1,4.333333,1.333333,19.333333,9.666667
111,2020-11-07,Sat,Bundesliga,Away,W,3.0,2.0,Dortmund,NaN,16.0,...,2021,Bayern Munich,5,0,5,1,3.666667,0.666667,14.333333,8.333333
127,2020-11-21,Sat,Bundesliga,Home,D,1.0,1.0,Werder Bremen,NaN,15.0,...,2021,Bayern Munich,5,1,22,0,3.333333,1.000000,14.333333,7.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2970,2025-04-19,Sat,Bundesliga,Away,W,4.0,0.0,Heidenheim,NaN,18.0,...,2025,Bayern Munich,5,0,10,1,2.666667,1.666667,22.000000,9.000000
2991,2025-04-26,Sat,Bundesliga,Home,W,3.0,0.0,Mainz,NaN,14.0,...,2025,Bayern Munich,5,1,16,1,3.000000,1.000000,21.666667,9.666667
3008,2025-05-03,Sat,Bundesliga,Away,D,3.0,3.0,RB Leipzig,NaN,20.0,...,2025,Bayern Munich,5,0,17,0,3.000000,0.666667,20.000000,9.333333
3026,2025-05-10,Sat,Bundesliga,Home,W,2.0,0.0,M'gladbach,NaN,24.0,...,2025,Bayern Munich,5,1,15,1,3.333333,1.000000,17.333333,7.666667


In [42]:
matches_rolling = combined_data.groupby("team").apply(lambda x: rolling_averages(x,cols,new_cols))

/var/folders/59/n670c_993y1708914z3l06zr0000gq/T/ipykernel_128/633759359.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = combined_data.groupby("team").apply(lambda x: rolling_averages(x,cols,new_cols))


In [43]:
matches_rolling

date  day        comp venue result   gf   ga  \
team                                                                 
Augsburg  54    2020-10-17  Sat  Bundesliga  Home      L  0.0  2.0   
          88    2020-10-26  Mon  Bundesliga  Away      L  1.0  3.0   
          92    2020-10-31  Sat  Bundesliga  Home      W  3.0  1.0   
          110   2020-11-07  Sat  Bundesliga  Home      L  0.0  3.0   
          126   2020-11-21  Sat  Bundesliga  Away      D  1.0  1.0   
...                    ...  ...         ...   ...    ...  ...  ...   
Wolfsburg 2981  2025-04-19  Sat  Bundesliga  Away      D  2.0  2.0   
          3001  2025-04-26  Sat  Bundesliga  Home      L  0.0  1.0   
          3017  2025-05-03  Sat  Bundesliga  Away      L  0.0  4.0   
          3025  2025-05-09  Fri  Bundesliga  Home      D  2.0  2.0   
          3059  2025-05-17  Sat  Bundesliga  Away      W  1.0  0.0   

                  opponent  referee    sh  ...  season       team  day_code  \
team                                       ...                                
Augsburg  54    RB Leipzig      NaN   6.0  ...    2021   Augsburg         5   
          88    Leverkusen      NaN   7.0  ...    2021   Augsburg         0   
          92         Mainz      NaN  14.0  ...    2021   Augsburg         5   
          110       Hertha      NaN  10.0  ...    2021   Augsburg         5   
          126   M'gladbach      NaN  10.0  ...    2021   Augsburg         5   
...                    ...      ...   ...  ...     ...        ...       ...   
Wolfsburg 2981       Mainz      NaN  16.0  ...    2025  Wolfsburg         5   
          3001    Freiburg      NaN   8.0  ...    2025  Wolfsburg         5   
          3017    Dortmund      NaN  11.0  ...    2025  Wolfsburg         5   
          3025  Hoffenheim      NaN  14.0  ...    2025  Wolfsburg         4   
          3059  M'gladbach      NaN  17.0  ...    2025  Wolfsburg         5   

                venue_code  opp_code  target  gf_rolling  ga_rolling  \
team                                                                   
Augsburg  54             1        17       0    1.666667    0.333333   
          88             0        14       0    0.666667    0.666667   
          92             1        16       1    0.333333    1.666667   
          110            1        11       0    1.333333    2.000000   
          126            0        15       0    1.333333    2.333333   
...                    ...       ...     ...         ...         ...   
Wolfsburg 2981           0        16       0    0.666667    1.666667   
          3001           1         8       0    1.333333    2.000000   
          3017           0         5       0    1.333333    2.000000   
          3025           1        12       0    0.666667    2.333333   
          3059           0        15       1    0.666667    2.333333   

               sh_rolling  sot_rolling  
team                                    
Augsburg  54     8.000000     3.333333  
          88     7.000000     2.333333  
          92     7.333333     2.000000  
          110    9.000000     4.000000  
          126   10.333333     4.000000  
...                   ...          ...  
Wolfsburg 2981  13.333333     3.666667  
          3001  15.000000     4.333333  
          3017  13.666667     3.666667  
          3025  11.666667     2.333333  
          3059  11.000000     2.666667  

[2982 rows x 27 columns]

In [44]:
matches_rolling = matches_rolling.droplevel('team')

In [45]:
matches_rolling

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
54,2020-10-17,Sat,Bundesliga,Home,L,0.0,2.0,RB Leipzig,NaN,6.0,...,2021,Augsburg,5,1,17,0,1.666667,0.333333,8.000000,3.333333
88,2020-10-26,Mon,Bundesliga,Away,L,1.0,3.0,Leverkusen,NaN,7.0,...,2021,Augsburg,0,0,14,0,0.666667,0.666667,7.000000,2.333333
92,2020-10-31,Sat,Bundesliga,Home,W,3.0,1.0,Mainz,NaN,14.0,...,2021,Augsburg,5,1,16,1,0.333333,1.666667,7.333333,2.000000
110,2020-11-07,Sat,Bundesliga,Home,L,0.0,3.0,Hertha,NaN,10.0,...,2021,Augsburg,5,1,11,0,1.333333,2.000000,9.000000,4.000000
126,2020-11-21,Sat,Bundesliga,Away,D,1.0,1.0,M'gladbach,NaN,10.0,...,2021,Augsburg,5,0,15,0,1.333333,2.333333,10.333333,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2981,2025-04-19,Sat,Bundesliga,Away,D,2.0,2.0,Mainz,NaN,16.0,...,2025,Wolfsburg,5,0,16,0,0.666667,1.666667,13.333333,3.666667
3001,2025-04-26,Sat,Bundesliga,Home,L,0.0,1.0,Freiburg,NaN,8.0,...,2025,Wolfsburg,5,1,8,0,1.333333,2.000000,15.000000,4.333333
3017,2025-05-03,Sat,Bundesliga,Away,L,0.0,4.0,Dortmund,NaN,11.0,...,2025,Wolfsburg,5,0,5,0,1.333333,2.000000,13.666667,3.666667
3025,2025-05-09,Fri,Bundesliga,Home,D,2.0,2.0,Hoffenheim,NaN,14.0,...,2025,Wolfsburg,4,1,12,0,0.666667,2.333333,11.666667,2.333333


In [46]:
matches_rolling.index = range(matches_rolling.shape[0])

In [47]:
matches_rolling

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
0,2020-10-17,Sat,Bundesliga,Home,L,0.0,2.0,RB Leipzig,NaN,6.0,...,2021,Augsburg,5,1,17,0,1.666667,0.333333,8.000000,3.333333
1,2020-10-26,Mon,Bundesliga,Away,L,1.0,3.0,Leverkusen,NaN,7.0,...,2021,Augsburg,0,0,14,0,0.666667,0.666667,7.000000,2.333333
2,2020-10-31,Sat,Bundesliga,Home,W,3.0,1.0,Mainz,NaN,14.0,...,2021,Augsburg,5,1,16,1,0.333333,1.666667,7.333333,2.000000
3,2020-11-07,Sat,Bundesliga,Home,L,0.0,3.0,Hertha,NaN,10.0,...,2021,Augsburg,5,1,11,0,1.333333,2.000000,9.000000,4.000000
4,2020-11-21,Sat,Bundesliga,Away,D,1.0,1.0,M'gladbach,NaN,10.0,...,2021,Augsburg,5,0,15,0,1.333333,2.333333,10.333333,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2977,2025-04-19,Sat,Bundesliga,Away,D,2.0,2.0,Mainz,NaN,16.0,...,2025,Wolfsburg,5,0,16,0,0.666667,1.666667,13.333333,3.666667
2978,2025-04-26,Sat,Bundesliga,Home,L,0.0,1.0,Freiburg,NaN,8.0,...,2025,Wolfsburg,5,1,8,0,1.333333,2.000000,15.000000,4.333333
2979,2025-05-03,Sat,Bundesliga,Away,L,0.0,4.0,Dortmund,NaN,11.0,...,2025,Wolfsburg,5,0,5,0,1.333333,2.000000,13.666667,3.666667
2980,2025-05-09,Fri,Bundesliga,Home,D,2.0,2.0,Hoffenheim,NaN,14.0,...,2025,Wolfsburg,4,1,12,0,0.666667,2.333333,11.666667,2.333333


In [48]:
def make_predictions(data,predictors):
    train = data[data["date"] < '2023-01-01']
    test = data[data["date"] > '2023-01-01']
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual = test["target"], predicted = preds), index = test.index)
    precision = precision_score(test["target"], preds)
    return combined, precision


In [49]:
combined, precision = make_predictions(matches_rolling, predictors + new_cols)


In [50]:
precision

0.5985915492957746

In [51]:
combined = combined.merge(matches_rolling[["date","team","opponent", "result"]], left_index=True, right_index=True)

In [53]:
class MissingDict(dict):
    __missing__ = lambda self, key: key

map_values = {"Bayern Munich": "Bayern Munich"}
mapping = MissingDict(**map_values)

In [54]:
combined["new_team"] = combined["team"].map(mapping)


In [55]:
combined

,actual,predicted,date,team,opponent,result,new_team
80,0,0,2023-01-22,Augsburg,Dortmund,L,Augsburg
81,1,0,2023-01-25,Augsburg,M'gladbach,W,Augsburg
82,0,0,2023-01-28,Augsburg,Freiburg,L,Augsburg
83,1,0,2023-02-03,Augsburg,Leverkusen,W,Augsburg
84,0,0,2023-02-11,Augsburg,Mainz,L,Augsburg
...,...,...,...,...,...,...,...
2977,0,0,2025-04-19,Wolfsburg,Mainz,D,Wolfsburg
2978,0,0,2025-04-26,Wolfsburg,Freiburg,L,Wolfsburg
2979,0,0,2025-05-03,Wolfsburg,Dortmund,L,Wolfsburg
2980,0,0,2025-05-09,Wolfsburg,Hoffenheim,D,Wolfsburg


In [56]:
merged = combined.merge(combined, left_on=["date", "new_team"], right_on=["date", "opponent"])


In [57]:
merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] ==0)]["actual_x"].value_counts()

actual_x
1    83
0    52
Name: count, dtype: int64

In [58]:
merged.to_csv("bundesligascsv/bundesligamerged.csv",index=False)

In [63]:
import pandas as pd
import numpy as np

POINTS_FOR_WIN = 3
POINTS_FOR_DRAW = 1
POINTS_FOR_LOSS = 0
GAMES_PER_BUNDESLIGA_SEASON = 38 
TOTAL_FIXTURES_PER_SEASON = 380 

DRAW_PROBABILITY_ON_ZERO = 0.25

def simulate_league_cumulative(df):
    print(f"--- Running Cumulative Simulation based on {len(df)} Fixtures ---")

    home_teams = df['team_x'].unique()
    away_teams = df['opponent_x'].unique()
    all_teams = sorted(list(set(list(home_teams) + list(away_teams))))

    league_table = {}
    for team in all_teams:
        league_table[team] = {
            'Points': 0,
            'Played': 0,
            'Wins': 0,
            'Draws': 0,
            'Losses': 0,        }

    for index, row in df.iterrows():
        home_team = row['team_x']
        away_team = row['opponent_x']
        prediction = int(row['predicted_x']) 

        league_table[home_team]['Played'] += 1
        league_table[away_team]['Played'] += 1

        if prediction == 1:
            league_table[home_team]['Points'] += POINTS_FOR_WIN
            league_table[home_team]['Wins'] += 1
            league_table[away_team]['Losses'] += 1

        elif prediction == 0:
           
            if np.random.rand() < DRAW_PROBABILITY_ON_ZERO:
                league_table[home_team]['Draws'] += 1
                league_table[home_team]['Points'] += POINTS_FOR_DRAW
                
                league_table[away_team]['Draws'] += 1
                league_table[away_team]['Points'] += POINTS_FOR_DRAW
            else:
                league_table[home_team]['Losses'] += 1
                
                league_table[away_team]['Points'] += POINTS_FOR_WIN
                league_table[away_team]['Wins'] += 1


    final_table = pd.DataFrame.from_dict(league_table, orient='index')
    final_table = final_table[final_table['Played'] > 0]
    
    return final_table


try:
    df_predictions = pd.read_csv('bundesligascsv/bundesligamerged.csv')

    df_predictions.columns = df_predictions.columns.str.strip().str.lower()
    
    required_cols_lower = ['team_x', 'opponent_x', 'predicted_x']
    if not all(col in df_predictions.columns for col in required_cols_lower):
        raise ValueError(f"CSV file must contain columns: {required_cols_lower}")

    cumulative_table = simulate_league_cumulative(df_predictions)

    if cumulative_table.empty:
        print("\nNo teams found in the prediction data.")
        raise SystemExit(0) 
        
    averaged_table = cumulative_table.copy()

    normalization_factor = averaged_table['Played'] / GAMES_PER_BUNDESLIGA_SEASON
    
    print(f"\nNOTE: Normalizing each team's statistics to an average single {GAMES_PER_BUNDESLIGA_SEASON}-match season.")

    cols_to_normalize = ['Wins', 'Draws', 'Losses', 'Points']
    
    for col in cols_to_normalize:
        new_col_name = f'Avg. {col}'
        averaged_table[new_col_name] = averaged_table[col].div(normalization_factor).fillna(0) 

    averaged_table['Avg. Played'] = float(GAMES_PER_PREMIER_LEAGUE_SEASON)
    final_cols = ['Avg. Played', 'Avg. Wins', 'Avg. Draws', 'Avg. Losses', 'Avg. Points']
    final_table = averaged_table[final_cols].copy()

    final_table = final_table.sort_values(
        by=['Avg. Points', 'Avg. Wins'], 
        ascending=[False, False]
    )

    final_table_display = final_table.round(2)
    
    max_played = cumulative_table['Played'].max()
    num_seasons_data = round(max_played / GAMES_PER_BUNDESLIGA_SEASON)
    if num_seasons_data < 1: num_seasons_data = 1

    print("\n" + "="*70)
    print(f"  PREDICTED BUNDESLIGA TABLE (Averaged over Data Span of {num_seasons_data} Seasons)")
    print("="*70)
    print(final_table_display.to_string())
    print("="*70)

    if not final_table.empty:
        print(f"\n--- Average Predicted Standings ---\n")
        avg_champ = final_table.index[0]
        avg_points = final_table_display['Avg. Points'].iloc[0] 
        print(f"Predicted Champion (Average Season): {avg_champ} with {avg_points} points.")
    
        if len(final_table) >= 20:
             relegation_teams = final_table.index[-3:].tolist()
             print(f"Predicted Relegation (Average Season): {', '.join(relegation_teams)}.")

        champions_league = final_table.index[:4].tolist()
        print(f"Predicted UEFA Champions League Group Stage (Average Season): {', '.join(champions_league)}.")

        europa_league= final_table.index[5]
        europa_league2 = final_table.index[6]
        print(f"Predicted Europa League Group Stage (Average Season): {europa_league}, {europa_league2}")
        

except FileNotFoundError:
    print(f"Error: The file 'merged.csv' was not found.")
    print("Please make sure 'merged.csv' is in the same directory as this script.")
except ValueError as e:
    print(f"Data Error: {e}")
except SystemExit:
    pass
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Running Cumulative Simulation based on 1530 Fixtures ---

NOTE: Normalizing each team's statistics to an average single 38-match season.

  PREDICTED BUNDESLIGA TABLE (Averaged over Data Span of 5 Seasons)
               Avg. Played  Avg. Wins  Avg. Draws  Avg. Losses  Avg. Points
Bayern Munich         38.0      24.97        6.41         6.63        81.30
Leverkusen            38.0      17.67        8.17        12.15        61.20
RB Leipzig            38.0      17.25        8.74        12.01        60.49
Stuttgart             38.0      16.82        7.43        13.76        57.87
Dortmund              38.0      15.24       10.83        11.93        56.56
Darmstadt             38.0      14.10        9.81        14.10        52.10
Schalke 04            38.0      15.00        7.00        16.00        52.00
Bochum                38.0      13.80        9.95        14.25        51.35
Ein Frankfurt         38.0      13.92        9.50        14.58        51.26
Werder Bremen         38.0    